# Agentic Tool-Call Loop: Questions 5–6

**Goal**: Implement a `while True` agentic loop that drives repeated model calls until the model stops requesting tools.

This notebook covers:
1. **Question 5**: Single tool call: ask the agent what the Capybara Wikipedia page is about
2. **Question 6**: Multi-step loop: ask about threats to capybara populations, requiring `search_wikipedia` followed by `get_page`

In [1]:
import json
import requests
from openai import OpenAI

## Wikipedia Tool Functions and Tool Schemas

Define both Python functions (`search_wikipedia`, `get_page`) and their OpenAI tool schemas: the JSON descriptions the model uses to decide which tool to call and with what arguments.

In [2]:
def search_wikipedia(query:str):
    """
    Search Wikipedia for articles matching a query string using the Wikipedia API.

    This function sends a GET request to the Wikipedia search API and returns
    the JSON response containing search results (titles, snippets, etc.).
    If the request fails, it catches the exception and returns None.

    Parameters:
    ----------
    query : str
        The search term or phrase to look up on Wikipedia.

    Returns:
    -------
    dict or None
        A dictionary containing the JSON response from the Wikipedia API if
        the request is successful. Returns None if the request fails.

    Example:
    -------
    >>> results = search_wikipedia("Python programming")
    >>> if results:
    ...     for item in results["query"]["search"]:
    ...         print(item["title"])
    ...
    Python (programming language)
    History of Python
    Python syntax and semantics
    """

    url = "https://en.wikipedia.org/w/api.php"
    
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}

    params = {
        "action":"query",
        "format":"json",
        "list":"search",
        "srsearch":query
    }
    try:
        response = requests.get(url=url, headers=headers, params=params, timeout=5)
        response.raise_for_status()
        json = response.json()
        return json
    except requests.exceptions.RequestException as e:
        print("Request failed:", e)
        return None

In [3]:
search_tool =  {
        "type": "function",
        "name": "search_wikipedia",
        "description": "Search Wikipedia for articles matching a query string and return search results (titles, snippets, and metadata).",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search term or phrase to look up on Wikipedia."
                }
            },
            "required": ["query"],
        },
    }

In [4]:
def get_page(page_title:str):
    """
    Retrieve the raw content of a Wikipedia page by its title.

    This function sends a GET request to Wikipedia and returns the raw
    wikitext (unformatted source content) of the specified page. If the
    request fails, it catches the exception and returns None.

    Parameters:
    ----------
    page_title : str
        The title of the Wikipedia page to retrieve.

    Returns:
    -------
    str or None
        A string containing the raw wikitext content of the page if the
        request is successful. Returns None if the request fails.

    Example:
    -------
    >>> content = get_page("Python (programming language)")
    >>> if content:
    ...     print(content[:500])  # print first 500 characters
    ...
    {{Short description|High-level programming language}}
    {{Use dmy dates|date=August 2021}}
    Python is an interpreted, high-level and general-purpose programming language...
    """

    url = "https://en.wikipedia.org/w/index.php"
    
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}

    params = {
        "title":page_title,
        "action":"raw"
    }
    try:
        response = requests.get(url=url, headers=headers, params=params, timeout=5)
        response.raise_for_status()
        text = response.text
        return text
    except requests.exceptions.RequestException as e:
        print("Request failed:", e)
        return None

In [5]:
get_page_tool = {
        "type": "function",
        "name": "get_page",
        "description": "Retrieve the raw wikitext content of a Wikipedia page by its title.",
        "parameters": {
            "type": "object",
            "properties": {
                "page_title": {
                    "type": "string",
                    "description": "The exact title of the Wikipedia page to retrieve (e.g., 'Python (programming language)')."
                }
            },
            "required": ["page_title"],
        },
    }

## `make_call`: Tool Dispatcher

Routes a `function_call` message from the model to the correct Python function. It parses the arguments, executes the function, and returns the result formatted as a `function_call_output` ready to be appended to the message history.

In [6]:
def make_call(tool_call):
    # let's parse the arguments
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == "get_page":
        ## Let's execute the search with those arguments
        result = get_page(**arguments)
    elif name == "search_wikipedia":
        result = search_wikipedia(**arguments)
    else:
        result = f'Not found tool {name}'

    # Format the result as JSON and create the function call output
    call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(result)
    }

    return call_output
        
    

In [9]:
instructions = """
You are a Wikipedia assistant.

Your job is to answer user questions using ONLY information RETRIEVED from a search on Wikipedia.

IMPORTANT: If you
+ Cannot retrieve from Wikipedia Web Page
+ Cannot find the answer from Wikipedia Web Page
YOU are forced to  NOT USE prior knowledge and say:
"I could not find the answer on Wikipedia Web Page."
"""

question = "What is this page about? https://en.wikipedia.org/wiki/Capybara"

tools = [search_tool, get_page_tool]

## Question 5: Single Tool Call

Initialise the OpenAI client, define the system instructions, and ask the agent what the Capybara Wikipedia page is about. The agent receives only the `get_page` tool and should call it to answer.

In [10]:
openai_client = OpenAI()

In [28]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools= [get_page_tool]
)

response

Response(id='resp_05e84363784b7c290069f20ab321d08196abc82fc5c20e0e02', created_at=1777470131.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFunctionToolCall(arguments='{"page_title":"Capybara"}', call_id='call_q0hYFcNUTvpbLmqbp83UPFOG', name='get_page', type='function_call', id='fc_05e84363784b7c290069f20ab440ec8196a96a5c15027ade1b', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='get_page', parameters={'type': 'object', 'properties': {'page_title': {'type': 'string', 'description': "The exact title of the Wikipedia page to retrieve (e.g., 'Python (programming language)')."}}, 'required': ['page_title'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Retrieve the raw wikitext content of a Wikipedia page by its title.')], top_p=1.0, background=False, completed_at=17774701

**Now extend the message history with the response output:**

In [29]:
message_history.extend(response.output)

In [43]:
for message in response.output:
    #print(message.type)
    if message.type == "function_call":
        tool_call_output = make_call(message)
        # Append the call output to our message history
        message_history.append(tool_call_output)
    if message.type == "message":
        text = message.content[0].text
        print(text)
    
        
        
        
        

The Wikipedia page about the **capybara** discusses this animal as the largest living rodent, native to all South American countries except Chile. Capybaras are semiaquatic herbivores that inhabit savannas and dense forests, primarily feeding on grasses and aquatic plants. They are highly social animals, often found in groups ranging from 10 to 20 individuals, but can also gather in larger numbers.

The page details the capybara's physical characteristics, behavior, diet, and mating practices. It also covers their ecological role, predation threats, and conservation status, noting that they are generally not considered threatened. Additionally, the article touches on human interactions with capybaras, including their farming for meat and skins, and their popularity in popular culture, particularly in Japan and on social media.


After the loop processes all tool calls, the message history contains the full conversation. Inspect the final text response from `response.output`.

**Getting the final response**

In [42]:
response.output[0].content[0].text

"The Wikipedia page about the **capybara** discusses this animal as the largest living rodent, native to all South American countries except Chile. Capybaras are semiaquatic herbivores that inhabit savannas and dense forests, primarily feeding on grasses and aquatic plants. They are highly social animals, often found in groups ranging from 10 to 20 individuals, but can also gather in larger numbers.\n\nThe page details the capybara's physical characteristics, behavior, diet, and mating practices. It also covers their ecological role, predation threats, and conservation status, noting that they are generally not considered threatened. Additionally, the article touches on human interactions with capybaras, including their farming for meat and skins, and their popularity in popular culture, particularly in Japan and on social media."

In [33]:
len(message_history)

4

In [32]:
print(response.output_text)

The Wikipedia page about the **capybara** discusses this animal as the largest living rodent, native to all South American countries except Chile. Capybaras are semiaquatic herbivores that inhabit savannas and dense forests, primarily feeding on grasses and aquatic plants. They are highly social animals, often found in groups ranging from 10 to 20 individuals, but can also gather in larger numbers.

The page details the capybara's physical characteristics, behavior, diet, and mating practices. It also covers their ecological role, predation threats, and conservation status, noting that they are generally not considered threatened. Additionally, the article touches on human interactions with capybaras, including their farming for meat and skins, and their popularity in popular culture, particularly in Japan and on social media.


In [35]:
response.output

[ResponseOutputMessage(id='msg_05e84363784b7c290069f20adf2a188196853245408785440e', content=[ResponseOutputText(annotations=[], text="The Wikipedia page about the **capybara** discusses this animal as the largest living rodent, native to all South American countries except Chile. Capybaras are semiaquatic herbivores that inhabit savannas and dense forests, primarily feeding on grasses and aquatic plants. They are highly social animals, often found in groups ranging from 10 to 20 individuals, but can also gather in larger numbers.\n\nThe page details the capybara's physical characteristics, behavior, diet, and mating practices. It also covers their ecological role, predation threats, and conservation status, noting that they are generally not considered threatened. Additionally, the article touches on human interactions with capybaras, including their farming for meat and skins, and their popularity in popular culture, particularly in Japan and on social media.", type='output_text', log

**Agentic Loop**

In [8]:
instructions = """
You are a Wikipedia assistant.

Your task is to answer the user’s question using ONLY information retrieved from Wikipedia pages.

You must follow this 3-step process:

1) First iteration:
   - Perform ONE search query on Wikipedia to find relevant pages.

2) Second iteration:
   - Analyze the results from the first search.
   - Then perform TWO additional, more refined search queries to improve relevance.

3) Final step:
   - Select the most relevant Wikipedia page(s).
   - Retrieve the content.
   - Synthesize the information into a final answer.

IMPORTANT:
- At EACH ITERATION, explain WHY you are performing the search query.
- Each explanation should be 2–3 sentences describing your reasoning.

STRICT RULES:
- Use ONLY Wikipedia pages obtained from your search results to provide answer.
- Do NOT use prior knowledge.
- PROVIDE THE LINK TO THE SOURCE AT THE END

FINAL ANSWER RULE:
- You MUST include a supporting quote or close paraphrase from the retrieved page for every statement.
- If you cannot provide supporting text, DO NOT include the statement.

FAILURE CONDITION:
If you cannot retrieve relevant Wikipedia pages or cannot find the answer in the retrieved content, respond exactly with:
"I COULD NOT FIND THE ANSWER on Wikipedia Web Page."

NOTE:
- The retrieved content may be in raw wikitext format. Ignore markup and extract only readable information.
"""

#question = "What are the main threats to capybara populations?"
question = "Who are the Beatles?"
tools = [search_tool, get_page_tool]

message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

## Question 6: Full Agentic Loop

Ask the agent `"What are the main threats to capybara populations?"`. The agent must search Wikipedia first, then fetch the relevant page combining both tools to reach an answer.

The `while True` loop runs until the model returns a response with no `function_call` messages. On each iteration:
- Execute any tool calls and append the results to the message history
- Break when the model returns a final text answer

In [11]:
iteration_number = 1

while True:
    response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=message_history,
    tools=tools,
    temperature=0.3
    )

    print(f'Iteration number {iteration_number}....')
    message_history.extend(response.output)

    has_function_call = False

    for message in response.output:
        #print(message.type)
        if message.type == "function_call":
            print(f"Executing {message.name}")
            tool_call_output = make_call(message)
            # Append the call output to our message history
            message_history.append(tool_call_output)
            has_function_call = True 
        if message.type == "message":
            text = message.content[0].text
            print("ASSISTANT:", text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_call:
        break
    

Iteration number 1....
ASSISTANT: I’ll start with a broad Wikipedia search for “Beatles” to identify the main page and any closely related pages. This first pass is meant to confirm the exact article title and see whether the search results point to the band or to a disambiguation page.
Executing search_wikipedia

Iteration number 2....
ASSISTANT: The first search clearly points to “The Beatles” as the main article, but it also shows related album and member pages. I’ll refine the search in two directions: one for the band’s identity and one for a concise summary of who they were, so I can answer directly from the most relevant Wikipedia text.
Executing search_wikipedia
Executing search_wikipedia

Iteration number 3....
ASSISTANT: I have enough from the refined searches to use the main “The Beatles” article, since it directly identifies the group and its members. I’ll retrieve that page and use its opening description, which should provide a concise, source-backed answer to who they ar

In [109]:
response.usage.input_tokens

15761